# Deep Reinforcement Lerning Lectures - Policy Gradients

### Imports and auxiliary settings

In [1]:
# -------- OS / Colab detection --------
import os
import sys
import platform
import subprocess

OS_NAME = platform.system()   # "Windows", "Linux", "Darwin"
# Force local mode
IN_COLAB = False

print("OS:", OS_NAME, "| Colab:", IN_COLAB)

# -------- Device selection --------
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


OS: Linux | Colab: True
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
107 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'm

In [76]:
%matplotlib inline

# PyTorch imports
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal
from torch.utils.data import Dataset, DataLoader

# Cross-framework library for DL
import onnx
from onnx2pytorch import ConvertModel

# Auxiliary Python imports
import math
import io
import base64
import random
import shutil
import copy
import glob
from time import time, strftime, sleep
import numpy as np
from tqdm.notebook import tqdm

# Environment imports
import gymnasium as gym
from gymnasium.spaces import Box
from gymnasium import logger as gymlogger
from gymnasium.wrappers import RecordVideo
gymlogger.min_level = gymlogger.ERROR

# Plotting and notebook imports
import matplotlib.pyplot as plt
import seaborn as sns; sns.set()
from IPython.display import HTML, display, clear_output

# -------- Virtual display (Linux / Colab only) --------
pydisplay = None

if OS_NAME == "Linux":
    try:
        from pyvirtualdisplay import Display
        pydisplay = Display(visible=0, size=(640, 480))
        pydisplay.start()
        print("Virtual display started.")
    except Exception as e:
        print("Skipping virtual display:", e)
else:
    print(f"Skipping virtual display on {OS_NAME} (not needed).")

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Virtual display started.
Device: cuda


In [77]:
class Logger(object):
    """Logger that can be used for debugging different values
    """
    def __init__(self, logdir, params=None, debug=False):
        self.gradients = []
        self.debug = debug
        self.basepath = os.path.join(logdir, strftime("%Y-%m-%dT%H-%M-%S"))
        os.makedirs(self.basepath, exist_ok=True)
        os.makedirs(self.log_dir, exist_ok=True)
        if params is not None and os.path.exists(params):
            shutil.copyfile(params, os.path.join(self.basepath, "params.pkl"))
        self.log_dict = {}
        self.dump_idx = {}

    def add_gradients(self, grad):
        if not self.debug: return
        self.gradients.append(grad)

    def compute_gradient_variance(self):
        vars_ = []
        grads_list = [np.zeros_like(self.gradients[0])] * 100
        for i, grads in enumerate(self.gradients):
            grads_list.append(grads)
            grads_list = grads_list[1:]
            grad_arr = np.stack(grads_list, axis=0)
            g = np.apply_along_axis(grad_variance, axis=-1, arr=grad_arr)
            vars_.append(np.mean(g))
        return vars_

    @property
    def param_file(self):
        return os.path.join(self.basepath, "params.pkl")

    @property
    def onnx_file(self):
        return os.path.join(self.basepath, "model.onnx")

    @property
    def video_dir(self):
        return os.path.join(self.basepath, "videos")

    @property
    def log_dir(self):
        return os.path.join(self.basepath, "logs")

    def log(self, name, value):
        if name not in self.log_dict:
            self.log_dict[name] = []
            self.dump_idx[name] = -1
        self.log_dict[name].append((len(self.log_dict[name]), time(), value))

    def get_values(self, name):
        if name in self.log_dict:
            return [x[2] for x in self.log_dict[name]]
        return None

    def dump(self):
        for name, rows in self.log_dict.items():
            with open(os.path.join(self.log_dir, name + ".log"), "a") as f:
                for i, row in enumerate(rows):
                    if i > self.dump_idx[name]:
                        f.write(",".join([str(x) for x in row]) + "\n")
                        self.dump_idx[name] = i

In [78]:
def wrap_env(env, logger, capture_video=True):
    # wrapper for recording
    env = gym.wrappers.RecordEpisodeStatistics(env)
    if capture_video:
        env = gym.wrappers.RecordVideo(env, logger.video_dir, episode_trigger=lambda idx: True)
    return env


def create_env(logger, env_id='BipedalWalker-v3', hardcore=False, capture_video=True):
    # initialize environment
    env = wrap_env(gym.make(env_id, max_episode_steps=1600, hardcore=hardcore, render_mode="rgb_array"), # Increased max_episode_steps
                   logger=logger,
                   capture_video=capture_video)
    action_size = env.action_space.shape[0]
    state_size = env.observation_space.shape[0]
    return env, action_size, state_size


def set_seed(env, seed=None):
    # seeding the envrionment
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)


def transforms(state):
    # transform to tensor and push to device, explicitly ensuring float32
    # Handle cases where `state` might already be a PyTorch tensor
    if not isinstance(state, torch.Tensor):
        state = torch.tensor(state, dtype=torch.float32)
    return state.to(device=device, dtype=torch.float32)


def test_environment(env, agent=None, seed=42, n_steps=200):
    # run and evaluate in the environment
    state, info = env.reset(seed=seed)
    for i in range(n_steps):
        env.render()

        if agent is None:
            action = env.action_space.sample()
        else:
            action, _ = agent.act(state)
            action = action.squeeze() # Removed .cpu().numpy() as agent.act already returns numpy arrays
        state, reward, done, truncated, info = env.step(action)
        if done:
            state, info = env.reset(seed=seed)
    env.close()


def get_running_stat(stat, stat_len):
    # evaluate stats
    cum_sum = np.cumsum(np.insert(stat, 0, 0))
    return (cum_sum[stat_len:] - cum_sum[:-stat_len]) / stat_len


def plot_results(runner):
    # plot stats
    episode, r, l = np.array(runner.stats_rewards_list).T
    cum_r = get_running_stat(r, 10)
    cum_l = get_running_stat(l, 10)

    plt.figure(figsize=(16, 16))

    plt.subplot(321)

    # plot rewards
    plt.plot(episode[-len(cum_r):], cum_r)
    plt.plot(episode, r, alpha=0.5)
    plt.xlabel('Episode')
    plt.ylabel('Episode Reward')

    plt.subplot(322)

    # plot episode lengths
    plt.plot(episode[-len(cum_l):], cum_l)
    plt.plot(episode, l, alpha=0.5)
    plt.xlabel('Episode')
    plt.ylabel('Episode Length')

    plt.subplot(323)

    # plot return
    all_returns = np.array(runner.buffer.all_returns)
    plt.scatter(range(0, len(all_returns)), all_returns, alpha=0.5)
    mean_returns = np.array(runner.buffer.mean_returns)
    plt.plot(range(0, len(mean_returns)), mean_returns, color="orange")
    plt.xlabel('Episode')
    plt.ylabel('Return')

    plt.subplot(324)

    # plot entropy
    entropy_arr = np.array(runner.stats_entropy_list)
    plt.plot(range(0, len(entropy_arr)), entropy_arr)
    plt.xlabel('Episode')
    plt.ylabel('Entropy')

    plt.subplot(325)

    if runner.logger.debug:
        # plot variance
        variance_arr = np.array(runner.logger.compute_gradient_variance())
        plt.plot(range(0, len(variance_arr)), variance_arr)
    plt.xlabel('Episode')
    plt.ylabel('Variance')

    plt.show()


"""
Utility functions to enable video recording of gym environment and displaying it
"""
def show_video(logger):
    print(logger.video_dir)
    mp4list = glob.glob(f'{logger.video_dir}/*.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0]
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        display(HTML(data='''<video alt="test" autoplay
                  loop controls style="height: 400px;">
                  <source src="data:video/mp4;base64,{0}" type="video/mp4" />
                </video>'''.format(encoded.decode('ascii'))))
    else:
        print("Could not find video")


def grad_variance(g):
    # compute gradient variance
    return np.mean(g**2) - np.mean(g)**2


class RunningMeanStd(object):
    """Tracks the mean and standard deviation of a stream of data.
    Used for observation normalization.
    """
    def __init__(self, epsilon=1e-4, shape=()):
        self.mean = np.zeros(shape, 'float64')
        self.var = np.ones(shape, 'float64')
        self.count = epsilon

    def update(self, x):
        batch_mean = np.mean(x, axis=0)
        batch_var = np.var(x, axis=0)
        batch_count = x.shape[0]
        self.update_from_moments(batch_mean, batch_var, batch_count)

    def update_from_moments(self, batch_mean, batch_var, batch_count):
        delta = batch_mean - self.mean
        tot_count = self.count + batch_count

        new_mean = self.mean + delta * batch_count / tot_count
        m_a = self.var * self.count
        m_b = batch_var * batch_count
        M2 = m_a + m_b + np.square(delta) * self.count * batch_count / tot_count
        new_var = M2 / tot_count
        new_count = tot_count

        self.mean = new_mean
        self.var = new_var
        self.count = new_count

    def normalize(self, obs):
        return (obs - self.mean) / np.sqrt(self.var + 1e-8)

### Test environment

In [79]:
# show behavior in envrionment with random agent
logger = Logger("logdir")
env, _, _ = create_env(logger)
test_environment(env)
show_video(logger)

logdir/2026-04-26T21-11-14/videos


In [80]:
class Transition(object):
    """Transition helper object
    """
    def __init__(self, state, action, reward, next_state, log_probs):
        self.state = state
        self.action = action
        self.reward = reward
        self.next_state = next_state
        self.log_probs = log_probs
        self.advantage = 0.0 # Will be filled by RolloutBuffer.compute_returns_and_advantages
        self.return_target = 0.0 # Will be filled by RolloutBuffer.compute_returns_and_advantages


class Episode(object):
    """Class for collecting an episode of transitions
    """
    def __init__(self, discount):
        self.discount = discount
        self._empty()
        self.total_reward = 0.0

    def _empty(self):
        self.n = 0
        self.transitions = []

    def reset(self):
        self._empty()

    def size(self):
        return self.n

    def append(self, transition):
        self.transitions.append(transition)
        self.n += 1

    def states(self):
        return torch.stack([transforms(s.state) for s in self.transitions])

    def actions(self):
        return torch.stack([transforms(a.action) for a in self.transitions])

    def rewards(self):
        return torch.tensor([r.reward for r in self.transitions], dtype=torch.float32, device=device)

    def next_states(self):
        return torch.stack([transforms(s_.next_state) for s_ in self.transitions])

    def log_probabilities(self):
        return torch.tensor([lp.log_probs for lp in self.transitions], dtype=torch.float32, device=device)


class BufferDataset(Dataset):
    """Buffer dataset used to iterate over buffer samples when training.
    """
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        t = self.data[idx]
        return t.state, t.action, t.log_probs, t.advantage, t.return_target


class RolloutBuffer(object):
    """Buffer to collect samples while rolling out in the environment.
    """
    def __init__(self, capacity, batch_size, min_transitions):
        self.capacity = capacity
        self.batch_size = batch_size
        self.min_transitions = min_transitions # Now represents rollout_steps
        self.episodes = []
        self.buffer = [] # This will be a flattened list of Transitions after GAE computation
        self.mean_returns = []
        self.all_returns = []
        self.current_num_transitions = 0

    def add(self, episode):
        # Saves an episode. GAE will be computed on these episodes later.
        self.episodes.append(episode)
        self.current_num_transitions += episode.size()

    def compute_returns_and_advantages(self, critic_net, discount, gae_lambda):
        self.buffer = [] # Clear the flattened buffer before re-populating
        for episode in self.episodes:
            states_tensor = episode.states().to(device)
            rewards_tensor = episode.rewards().to(device)

            with torch.no_grad():
                # Get value estimates for all states in the episode
                # Ensure states_tensor is float and correct shape for critic
                values = critic_net(states_tensor.float()).squeeze(1)

            advantages = torch.zeros_like(rewards_tensor, dtype=torch.float32, device=device)
            returns_target = torch.zeros_like(rewards_tensor, dtype=torch.float32, device=device)

            last_gae_lambda = 0
            # Iterate backwards to compute GAE and returns
            for i in reversed(range(len(rewards_tensor))):
                current_state_value = values[i]
                next_state_value = values[i+1] if i + 1 < len(rewards_tensor) else 0.0 # Next state value is 0 if it's the last step

                # TD-error
                delta = rewards_tensor[i] + discount * next_state_value - current_state_value
                # GAE calculation
                last_gae_lambda = delta + discount * gae_lambda * last_gae_lambda
                advantages[i] = last_gae_lambda
                returns_target[i] = advantages[i] + current_state_value

            # Store computed advantages and return targets into transitions
            for i, transition in enumerate(episode.transitions):
                transition.advantage = advantages[i].item()
                transition.return_target = returns_target[i].item()
                self.buffer.append(transition)

        self.episodes = [] # Clear episodes after processing them into the buffer
        self.current_num_transitions = len(self.buffer)

    def update_stats(self):
        # update the statistics on the buffer
        # Use return_target for stats, as it is the actual target for value function.
        returns = [t.return_target for t in self.buffer]
        if returns:
            self.all_returns += returns
            mean_return = np.mean(np.array(returns))
            self.mean_returns += ([mean_return]*len(returns))

    def reset(self):
        # calls empty
        self.episodes = []
        del self.buffer[:]
        self.current_num_transitions = 0

    def create_dataloader(self):
        # creates a dataloader for training
        train_loader = DataLoader(
            BufferDataset(self.buffer),
            batch_size=self.batch_size,
            shuffle=True,
            drop_last=True  # CRITICAL: prevents 1-sample batches where std()=NaN
        )
        return train_loader

    def __len__(self):
        return self.current_num_transitions # Report the number of collected transitions

In [87]:
class ActorNet(nn.Module):
    """Actor network (policy)"""
    def __init__(self, state_size, action_size, hidden_size):
        super(ActorNet, self).__init__()
        self.fc1 = nn.Linear(state_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.mu_head = nn.Linear(hidden_size, action_size)
        self.log_std = nn.Parameter(torch.zeros(action_size))

    def forward(self, x):
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        mu = self.mu_head(x)
        # Mandatory clamp for stability
        log_std = torch.clamp(self.log_std, -3.0, -1.0)
        return mu, log_std.expand_as(mu)

class CriticNet(nn.Module):
    """Critic network computing the state value"""
    def __init__(self, state_size, action_size, hidden_size):
        super(CriticNet, self).__init__()
        self.fc1 = nn.Linear(state_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.value_head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        value = self.value_head(x)
        return value

class ActorCriticNet(nn.Module):
    def __init__(self, state_size, action_size, actor_hidden_size, critic_hidden_size):
        super(ActorCriticNet, self).__init__()
        self.actor = ActorNet(state_size, action_size, actor_hidden_size)
        self.critic = CriticNet(state_size, action_size, critic_hidden_size)

    def forward(self, x):
        return self.actor(x)

    def act(self, state):
        mu_logits, log_std = self.actor(state)
        mu = torch.tanh(mu_logits)
        std = torch.exp(log_std)
        dist = Normal(mu, std + 1e-6) # Added epsilon for stability
        action = dist.sample()
        log_prob = dist.log_prob(action).sum(axis=-1)
        action = torch.clamp(action, -1.0, 1.0)
        return action.cpu().numpy(), log_prob.cpu().numpy()

    def evaluate(self, state, action):
        mu_logits, log_std = self.actor(state)
        mu = torch.tanh(mu_logits)
        std = torch.exp(log_std)
        dist = Normal(mu, std + 1e-6)
        log_prob = dist.log_prob(action).sum(axis=-1)
        entropy = dist.entropy().sum(axis=-1)
        state_value = self.critic(state)
        return log_prob, entropy, state_value

### Define Agent

In [82]:
class Agent(object):
    """Agent class used for training, saving data and handling the model.
    """
    def __init__(self, buffer, state_size, action_size, actor_hidden_size, critic_hidden_size, actor_lr, critic_lr, logger, eps_clip, n_epochs,
                 weight_decay, betas, loss_scales, discount, gae_lambda, obs_normalizer, checkpoint_dir="ckpts"):
        self.action_size = action_size
        self.state_size = state_size
        self.buffer = buffer
        self.checkpoint_dir = checkpoint_dir
        self.loss_scales = loss_scales # (policy_coeff, value_coeff, entropy_coeff)
        self.n_epochs = n_epochs
        self.eps_clip = eps_clip
        self.logger = logger
        self.discount = discount
        self.gae_lambda = gae_lambda
        self.obs_normalizer = obs_normalizer # Added observation normalizer
        # ===================================================
        # ++++++++++++++++ YOUR CODE HERE +++++++++++++++++++
        # ===================================================
        # 1) initialize ActorCriticNet instance
        # 2) create initializers
        # ===================================================
        self.model = ActorCriticNet(state_size, action_size, actor_hidden_size, critic_hidden_size).to(device)
        self.actor_optimizer = optim.Adam(self.model.actor.parameters(), lr=actor_lr, betas=betas, eps=1e-5, weight_decay=weight_decay)
        self.critic_optimizer = optim.Adam(self.model.critic.parameters(), lr=critic_lr, betas=betas, eps=1e-5, weight_decay=weight_decay)


    def save_checkpoint(self, epoch, info=''):
        """Saves a model checkpoint"""
        state = {
            'info': info,
            'epoch': epoch,
            'state_dict': self.model.state_dict(),
            'actor_optimizer': self.actor_optimizer.state_dict(),
            'critic_optimizer': self.critic_optimizer.state_dict()
        }
        os.makedirs(self.checkpoint_dir, exist_ok=True)
        ckp_name = 'best-checkpoint.pth' if info == 'best' else f'checkpoint-epoch{epoch}.pth'
        filename = os.path.join(self.checkpoint_dir, ckp_name)
        torch.save(state, filename)

    def resume_checkpoint(self, resume_path):
        """Resumes training from an existing model checkpoint"""
        print("Loading checkpoint: {} ...".format(resume_path))
        checkpoint = torch.load(resume_path)
        # load architecture params from checkpoint.
        self.model.load_state_dict(checkpoint['state_dict'])
        # load optimizer state from checkpoint only when optimizer type is not changed.
        self.actor_optimizer.load_state_dict(checkpoint['actor_optimizer'])
        self.critic_optimizer.load_state_dict(checkpoint['critic_optimizer'])
        print("Checkpoint loaded. Resume training")

    def load_onnx_checkpoint(self, onnx_file):
        self.model.actor = ConvertModel(onnx.load(onnx_file)).to(device)
        self.model.eval()

    def save_onnx_checkpoint(self):
        """Create an ONNX checkpoint"""
        dummy_input = torch.randn((1, self.state_size))
        dummy_input_t = transforms(dummy_input).to(device)

        model = self.model.to(device)
        model.eval() # Set model to evaluation mode

        # Export the full ActorCriticNet, but its forward is just the actor's part.
        # Export to a temporary file first
        temp_onnx_path = os.path.join(self.checkpoint_dir, "temp_submission_actor.onnx")
        final_onnx_path = os.path.join(self.checkpoint_dir, "submission_actor.onnx")

        # Ensure the forward method of ActorCriticNet returns what ONNX expects for the actor (mu_logits, log_std)
        # The `forward` method in ActorCriticNet is modified to return `mu` and `log_std` directly.
        torch.onnx.export(model, dummy_input_t, temp_onnx_path,
                         verbose=False, opset_version=18, export_params=True, do_constant_folding=True,
                         input_names=['input'], output_names=['mu_logits', 'log_std'])

        # Load the exported model and save it again, ensuring all weights are embedded
        loaded_model = onnx.load(temp_onnx_path)
        onnx.save(loaded_model, final_onnx_path, save_as_external_data=False)

        # Remove the temporary file if it exists
        if os.path.exists(temp_onnx_path):
            os.remove(temp_onnx_path)

        self.check(final_onnx_path)

    def check(self, file_name):
        # Load the ONNX model
        model = onnx.load(file_name)
        # Check that the IR is well formed
        onnx.checker.check_model(model)

    def act(self, state):
        # ===================================================
        # ++++++++++++++++ YOUR CODE HERE +++++++++++++++++++
        # ===================================================
        # 1) prepare state tensors
        # 2) check if shape is ok or expand acordingly
        # 3) get action and log probabilities
        # ===================================================
        # get action probs then sample from the probabilities
        # Normalize state before feeding to actor
        normalized_state = self.obs_normalizer.normalize(state)
        state_t = transforms(normalized_state).unsqueeze(0) if normalized_state.ndim == 1 else transforms(normalized_state)
        with torch.no_grad():
            mu_logits, log_std_logits = self.model.actor(state_t)
            mu = torch.tanh(mu_logits)
            std = torch.exp(log_std_logits)
            dist = Normal(mu, std)
            action = dist.sample()
            log_prob = dist.log_prob(action).sum(axis=-1)

        # Clamp the action to environment's bounds [-1, 1]
        action = torch.clamp(action, -1.0, 1.0)

        return action.cpu().numpy(), log_prob.cpu().numpy()

    def train(self):
        self.model.train() # Set model to training mode

        total_losses = []
        value_losses = []
        policy_losses = []
        entropies = []

        # Calculate GAE and returns for the entire buffer before training epochs
        # This ensures fresh and consistent targets for all updates in the epochs.
        self.buffer.compute_returns_and_advantages(self.model.critic, self.discount, self.gae_lambda)

        for i in range(self.n_epochs):
            # create the a dataloader based on the current buffer
            loader = self.buffer.create_dataloader()
            # iterate over the samples in the dataloader
            for states, actions, old_log_probs, advantages, returns_target in loader:
                # Normalize states from the dataloader (which stores raw states) before evaluation
                normalized_states = self.obs_normalizer.normalize(states.cpu().numpy()) # Convert to numpy for normalizer
                states_t = transforms(normalized_states)

                actions_t = transforms(actions)
                old_log_probs_t = transforms(old_log_probs).detach() # Ensure old_log_probs are detached
                advantages_t = transforms(advantages)
                returns_target_t = transforms(returns_target).unsqueeze(1) # Ensure shape matches state_values

                # Get current values and log probabilities from the model using normalized states
                log_probs, entropy, state_values = self.model.evaluate(states_t, actions_t)

                # Normalize advantages
                advantages_t = (advantages_t - advantages_t.mean()) / (advantages_t.std(correction=0) + 1e-8)

                # Calculate Ratio
                ratio = torch.exp(log_probs.unsqueeze(1) - old_log_probs_t.unsqueeze(1))

                # Clipped Surrogate Objective
                surrogate_loss1 = ratio * advantages_t
                surrogate_loss2 = torch.clamp(ratio, 1 - self.eps_clip, 1 + self.eps_clip) * advantages_t
                policy_loss = -torch.min(surrogate_loss1, surrogate_loss2).mean()

                # Value Loss (MSE)
                value_loss = F.mse_loss(state_values, returns_target_t)

                # Entropy Loss
                entropy_loss = -entropy.mean() # Negative sign because we want to maximize entropy

                # Total Loss for policy and value separately
                # policy_coeff, value_coeff, entropy_coeff
                actor_loss = self.loss_scales[0] * policy_loss + self.loss_scales[2] * entropy_loss # Entropy is usually added (maximized)
                critic_loss = self.loss_scales[1] * value_loss

                # Optimization step for actor
                self.actor_optimizer.zero_grad()
                actor_loss.backward()
                nn.utils.clip_grad_norm_(self.model.actor.parameters(), max_norm=0.5) # max_norm is 0.5 per requirement
                self.actor_optimizer.step()

                # Optimization step for critic
                self.critic_optimizer.zero_grad()
                critic_loss.backward()
                nn.utils.clip_grad_norm_(self.model.critic.parameters(), max_norm=0.5) # max_norm is 0.5 per requirement
                self.critic_optimizer.step()

                total_loss = actor_loss.item() + critic_loss.item() # For logging purposes
                total_losses.append(total_loss)
                value_losses.append(value_loss.item())
                policy_losses.append(policy_loss.item())
                entropies.append(entropy.mean().item())

        # return losses and entropy
        return (np.mean(total_losses), np.mean(value_losses), np.mean(policy_losses)), np.mean(entropies)

### Define Task Runner

In [90]:
# Final optimized training loop
ENV_ID = "BipedalWalker-v3"
NUM_ENVS = 16
ROLLOUT_STEPS = 2048
MINI_BATCH_SIZE = 256
TOTAL_TIMESTEPS = 5_000_000
LR_ACTOR = 3e-4
LR_CRITIC = 1e-3
GAMMA = 0.99
GAE_LAMBDA = 0.95
CLIP_EPS = 0.2
ENT_COEF = 0.001
VF_COEF = 0.5
MAX_GRAD_NORM = 0.5

envs = gym.vector.SyncVectorEnv([lambda: gym.make(ENV_ID) for _ in range(NUM_ENVS)])
obs_dim = envs.single_observation_space.shape[0]
act_dim = envs.single_action_space.shape[0]

actor = Actor(obs_dim, act_dim).to(device)
critic = Critic(obs_dim).to(device)
optimizer = optim.Adam([
    {"params": actor.parameters(), "lr": LR_ACTOR},
    {"params": critic.parameters(), "lr": LR_CRITIC},
], eps=1e-5)

buffer = RolloutBuffer(ROLLOUT_STEPS, NUM_ENVS, obs_dim, act_dim, device)
obs_rms = RunningMeanStd(shape=(obs_dim,))
ret_rms = RunningMeanStd(shape=())
returns_running = np.zeros(NUM_ENVS)

obs_np, _ = envs.reset()
global_step = 0
num_updates = TOTAL_TIMESTEPS // (ROLLOUT_STEPS * NUM_ENVS)

best_reward = -np.inf

for update in range(1, num_updates + 1):
    frac = 1.0 - (update - 1) / num_updates
    optimizer.param_groups[0]["lr"] = LR_ACTOR * frac
    optimizer.param_groups[1]["lr"] = LR_CRITIC * frac
    
    buffer.reset()
    for step in range(ROLLOUT_STEPS):
        global_step += NUM_ENVS
        obs_rms.update(obs_np)
        obs_norm = torch.FloatTensor(obs_rms.normalize(obs_np)).to(device)
        with torch.no_grad():
            action, log_prob, _ = actor.get_action_and_log_prob(obs_norm)
            value = critic(obs_norm)
        
        next_obs_np, rewards_np, terms_np, truncs_np, _ = envs.step(np.clip(action.cpu().numpy(), -1, 1))
        dones_np = (terms_np | truncs_np).astype(np.float32)
        
        returns_running = returns_running * GAMMA + rewards_np
        ret_rms.update(returns_running)
        rewards_norm = rewards_np / np.sqrt(ret_rms.var + 1e-8)
        
        for i, d in enumerate(dones_np):
            if d: returns_running[i] = 0.0
            
        buffer.store(obs_norm, action, log_prob, torch.FloatTensor(rewards_norm).to(device), torch.FloatTensor(dones_np).to(device), value)
        obs_np = next_obs_np

    with torch.no_grad():
        obs_norm = torch.FloatTensor(obs_rms.normalize(obs_np)).to(device)
        last_values = critic(obs_norm)
    buffer.compute_gae(last_values, torch.FloatTensor(dones_np).to(device), GAMMA, GAE_LAMBDA)
    
    ppo_update(actor, critic, optimizer, buffer, CLIP_EPS, VF_COEF, ENT_COEF, MAX_GRAD_NORM, 10, MINI_BATCH_SIZE)
    
    if update % 5 == 0:
        eval_rew = evaluate_policy(actor, obs_rms, ENV_ID)
        print(f"Update {update}/{num_updates} | Step {global_step} | Eval Reward: {eval_rew:.2f}")
        if eval_rew > best_reward:
            best_reward = eval_rew
            torch.save({"actor": actor.state_dict(), "obs_rms_mean": obs_rms.mean, "obs_rms_var": obs_rms.var}, "best_model.pt")
            if best_reward > 250:
                print("Target reached!")
                break


In [ ]:
plot_results(runner)

In [ ]:
rewards = [r for _, r, _ in runner.stats_rewards_list]
average_reward = np.mean(rewards)
print(f"Average reward score across all episodes: {average_reward:.2f}")

In [ ]:
# Cell 1: load best checkpoint
agent.resume_checkpoint('ckpts/best-checkpoint.pth')


In [ ]:
agent.save_onnx_checkpoint()

In [ ]:
from google.colab import files

# Assuming the ONNX file is saved in the 'ckpts' directory as 'submission_actor.onnx'
onnx_file_path = os.path.join(agent.checkpoint_dir, "submission_actor.onnx")
onnx_data_file_path = onnx_file_path + ".data"

if os.path.exists(onnx_file_path):
    files.download(onnx_file_path)
    print(f"Downloaded {onnx_file_path}")

    if os.path.exists(onnx_data_file_path):
        files.download(onnx_data_file_path)
        print(f"Downloaded {onnx_data_file_path}")
    else:
        print(f"Warning: ONNX data file not found at {onnx_data_file_path}. This might be expected if the model is small.")
else:
    print(f"Error: ONNX file not found at {onnx_file_path}")

## Visualize Agent

In [ ]:
c# load model from checkpoint
agent.resume_checkpoint('ckpts/best-checkpoint.pth')

In [ ]:
# run agent in the envrionment
env, _, _ = create_env(logger)
agent.model.eval()
test_environment(env=env, agent=agent)
show_video(logger)

# Demo Evaluations

In [ ]:
agent.load_onnx_checkpoint('demo.onnx')

In [ ]:
# run agent in the envrionment
logger = Logger('logdir')
env, _, _ = create_env(logger)
agent.model.eval()
test_environment(env=env, agent=agent, n_steps=500)
show_video(logger)